In [1]:
import os, sys
import numpy as np
import pandas as pd
import re
#import dill
from sklearn.metrics import roc_auc_score, f1_score, root_mean_squared_error
import random
from datetime import date
import time
parts=os.getcwd().split('/')[:-1]
code_path='/'.join(parts)+'/scripts/'
sys.path.append(code_path)
%load_ext autoreload

In [2]:
#!pip install -U git+https://github.com/rickvanveen/sklvq.git


In [3]:
from sklvq import GLVQ, GMLVQ
import sklvq
datapath='/'.join(parts)+'/data/'

In [4]:

dname='breastcancer'
trainset, testset=pd.read_csv(datapath+'%s_trainset.csv'%dname), pd.read_csv(datapath+'%s_testset.csv'%dname)
#Xtrain, Ytrain=
features=trainset.columns[:-1]
Xtrain, Ytrain=trainset[features], trainset['Label'].to_numpy()
Xtest, Ytest=testset[features], testset['Label'].to_numpy()
mu, std=Xtrain.mean(), Xtrain.std()
zXtrain, zXtest=(Xtrain-mu)/std, (Xtest-mu)/std
zXtrain.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,0.355353,-1.724380,0.412489,0.206163,2.147466,1.002636,1.445146,1.591784,0.594346,1.104287,...,0.510357,-1.408001,0.521108,0.373444,0.919903,0.722023,1.569338,1.055136,0.517762,0.987956
1,-0.372777,0.508145,-0.394182,-0.403978,-1.486005,-0.396129,-0.351068,-0.788586,-0.828195,-0.209696,...,-0.391717,0.241067,-0.472863,-0.413526,-1.629899,-0.370711,-0.432086,-0.895497,-0.631537,-0.114546
2,1.401515,-0.151251,1.346229,1.344966,0.670622,0.445760,0.547925,1.075074,0.413764,-0.420209,...,2.020664,0.427973,1.864474,2.172676,1.321276,0.674254,0.591187,1.461614,1.024527,-0.033654


In [5]:
glvq=model = GLVQ(
    distance_type="squared-euclidean", activation_type="identity", 
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    random_state=42)
glvq_copy=GLVQ(
    distance_type="squared-euclidean", activation_type="identity",
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    random_state=42)
glvq.fit(zXtrain, Ytrain)
glvq_copy.fit(zXtrain, Ytrain)
glvq.prototypes_secure=glvq_copy.prototypes_.copy()
est_train, est_test=glvq.predict(zXtrain), glvq.predict(zXtest)
est_train_copy, est_test_copy=glvq_copy.predict(zXtrain), glvq_copy.predict(zXtest)

## Unlearn sample influences

In [7]:
unlearn_indices=np.random.randint(0, trainset.shape[0] + 1, size=30).tolist()
relearn_indices=np.setdiff1d(list(range(0, trainset.shape[0])), unlearn_indices)
unlearn_samples=trainset[features].iloc[unlearn_indices].copy()
relearn_samples=trainset[features].iloc[relearn_indices].copy()
unlearn_samples2=unlearn_samples.copy()
relearn_samples2=relearn_samples.copy()
unlearn_samples2.insert(0,'og_index',unlearn_indices)
unlearn_samples2.to_csv(datapath+'unlearn_samples.csv', index=False)
relearn_samples2.insert(0,'og_index',relearn_indices)
relearn_samples2.to_csv(datapath+'relearn_samples.csv', index=False)
#print(unlearn_indices)
#print(relearn_indices)
glvq_partial1=GLVQ(distance_type="squared-euclidean", activation_type="identity", solver_type="wgd", 
                solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},random_state=42)
mu_new, std_new= relearn_samples.mean(), relearn_samples.std()
zXretrain, zXretest=(relearn_samples-mu_new)/std_new, (Xtest-mu_new)/std_new
glvq_partial1.fit(zXretrain, Ytrain[relearn_indices])
Ytrain[unlearn_indices]

array([0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 0])

In [8]:
def unlearn_sample_effect_glvq2(model, labels_unlearn, data_unlearn):
    updated_prots=model.prototypes_.copy()#np.empty((np.shape(model.prototypes_)))*0
    dist_func=sklvq.distances.SquaredEuclidean()
    computed_dist=dist_func(data_unlearn, model)
    max_runs, k=model.get_params()['solver_params']['max_runs'], model.get_params()['solver_params']['k']
    step_size=model.get_params()['solver_params']['step_size'][0]
    previous_waypoints = np.empty((k, model.get_variables().size))
    tentative_model_variables = np.empty(model.get_variables().size)

    for iter in range(0, k):
        for nprot in model.prototypes_labels_:
            get_grad=sklvq.distances.SquaredEuclidean.gradient(sklvq.distances.SquaredEuclidean(), 
                                                               data=data_unlearn, model=model, i_prototype=nprot)
            if nprot==0:
                dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot+1]
            else:
                dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot-1]
            prot=np.reshape(model.prototypes_[nprot], (1,model.prototypes_.shape[1]))
            ind_same,ind_diff=np.where(dist_same<dist_diff)[0], np.where(dist_same>dist_diff)[0]
            if nprot>=0:
                if len(ind_same)>=1:            
                    if len(ind_same)==1:
                        unlearn_grad_same=get_grad.iloc[ind_same].copy()*data_unlearn.iloc[ind_same]
                        prot_init_corr=np.reshape(data_unlearn.iloc[ind_same], (1,model.prototypes_.shape[1]))
                    else:
                        unlearn_grad_same=get_grad.iloc[ind_same].mean()* data_unlearn.iloc[ind_same].mean()
                        prot_init_corr=np.reshape(data_unlearn.iloc[ind_same].mean(), (1,model.prototypes_.shape[1]))
                else:
                    unlearn_grad_same=np.zeros(np.shape(prot))
                if len(ind_diff)>=1:
                    if len(ind_diff)==1:
                        unlearn_grad_diff=get_grad.iloc[ind_diff].copy()*data_unlearn.iloc[ind_diff]
                        prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff], (1,model.prototypes_.shape[1]))
                    else:
                        unlearn_grad_diff=get_grad.iloc[ind_diff].mean()*data_unlearn.iloc[ind_diff].mean()
                        prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff].mean(), (1,model.prototypes_.shape[1]))
                   # prot-=prot_init_corr
                else:
                    unlearn_grad_diff=np.zeros(np.shape(prot))
                unlearn_grad_diff=np.reshape(unlearn_grad_diff, (1,model.prototypes_.shape[1]))
                unlearn_grad_same=np.reshape(unlearn_grad_same, (1,model.prototypes_.shape[1]))
                updated_prots[nprot,:]=updated_prots[nprot,:]+(unlearn_grad_diff-unlearn_grad_same)*step_size
        model.set_prototypes(updated_prots)
    return model

In [9]:

#from unlearning.unlearn_lvq import unlearn_sample_effect_glvq
updated_model=unlearn_sample_effect_glvq2(glvq, Ytrain[unlearn_indices], zXtrain.iloc[unlearn_indices])

#glvq_copy.prototypes_=updated_prots
print(updated_model.prototypes_)
print(glvq_partial1.prototypes_)
print(np.mean(np.abs(updated_model.prototypes_-glvq_partial1.prototypes_)))
print(root_mean_squared_error(updated_model.prototypes_, glvq_partial1.prototypes_))
print(np.shape(updated_model.prototypes_), np.shape(glvq_partial1.prototypes_))

[[ 1.04948238 -0.15612689  1.10372059  1.01378456  0.54668417  1.04995943
   1.07790965  1.32134777  0.55267961 -0.06114486  0.72772397 -0.03762937
   0.7427724   0.7425163  -0.09328911  0.44935605  0.42303383  0.62347112
  -0.1344694   0.07369701  1.02998002  0.0357007   1.04014135  0.94379072
  -0.38508148  1.02984286  0.92540263  1.41821093  0.78955527  0.37630787]
 [-0.69387466 -0.55825232 -0.70423176 -0.63088486 -0.40598562 -0.52478157
  -0.61481589 -0.6815045  -0.42563027 -0.04561629 -0.43635364 -0.04266811
  -0.43605751 -0.42510241 -0.17229486 -0.28260392 -0.24583476 -0.36915201
  -0.03485799 -0.09062588 -0.70200191 -0.8405485  -0.70972147 -0.62934666
  -0.51990338 -0.71709111 -0.6827765  -0.79753103 -0.46437213 -0.36213537]]
[[ 0.99231765  0.55934424  1.01020801  0.96112549  0.43989206  0.78050927
   0.94093007  1.07368505  0.45986531 -0.11309517  0.73645484  0.00150036
   0.71580506  0.73269683 -0.10237424  0.34698049  0.31096792  0.50509701
  -0.00926649  0.04934114  1.058564

In [11]:
%autoreload 2
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq
updated_model=unlearn_sample_effect_glvq(glvq, Ytrain[unlearn_indices], zXtrain.iloc[unlearn_indices])

#glvq_copy.prototypes_=updated_prots
print(updated_model.prototypes_)
print(glvq_partial1.prototypes_)
print(np.mean(np.abs(updated_model.prototypes_-glvq_partial1.prototypes_)))
print(root_mean_squared_error(updated_model.prototypes_, glvq_partial1.prototypes_))
print(np.shape(updated_model.prototypes_), np.shape(glvq_partial1.prototypes_))

ValueError: XA must be a 2-dimensional array.

## GLVQ with 2 prototypes

In [25]:
glvq2=model = GLVQ(
    distance_type="squared-euclidean", activation_type="identity", 
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    prototype_n_per_class=2,
    random_state=42)
glvq2_copy=GLVQ(
    distance_type="squared-euclidean", activation_type="identity",
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    prototype_n_per_class=2,
    random_state=42)
glvq2.fit(zXtrain, Ytrain)
glvq2_copy.fit(zXtrain, Ytrain)
est2_train, est2_test=glvq2.predict(zXtrain), glvq2.predict(zXtest)
est2_train_copy, est2_test_copy=glvq2_copy.predict(zXtrain), glvq2_copy.predict(zXtest)
glvq2_partial1=GLVQ(distance_type="squared-euclidean", activation_type="identity", solver_type="wgd", 
                prototype_n_per_class=2,
                solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},random_state=42)
glvq2_partial1.fit(zXretrain, Ytrain[relearn_indices])
est2_train, est2_test=glvq2.predict(zXtrain), glvq2.predict(zXtest)
est2_train_copy, est2_test_copy=glvq2_copy.predict(zXtrain), glvq2_copy.predict(zXtest)
est2_partial_retrain, est2_partial_retest=glvq2_partial1.predict(zXretrain), glvq2_partial1.predict(zXretest)

In [26]:
%autoreload 2
#np.empty((np.shape(model.prototypes_)))*0
import scipy
from utils import compute_distances

dist_same, dist_diff, i_dist_same, i_dist_diff=compute_distances(glvq2, zXtrain.iloc[unlearn_indices], Ytrain[unlearn_indices])
print(i_dist_same[:5])
print(i_dist_diff[:5])

[3 0 1 2 2]
[0 2 2 0 0]


In [27]:

#sorted_computed_dist=np.argsort(computed_dist2, axis=1, kind='stable')[:,:2]#.T
#ind_same, ind_diff=computed_dist2[:,0], sorted_computed_dist[:,1]
#dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot-1]
#yind, xind_same=np.where(sorted_computed:,_dist==0)
#yind, xind_diff=np.where(sorted_computed_dist==1)
#nprots=glvq2.prototypes_.shape
#ind_same
data_unlearn=zXtrain.iloc[unlearn_indices].copy()
updated_prots=glvq2_copy.prototypes_.copy()
step_size=glvq2.get_params()['solver_params']['step_size'][0]
for nprot in range(0,glvq2.prototypes_labels_.shape[0]):
    prot=np.reshape(glvq2.prototypes_[nprot], (1,glvq2.prototypes_.shape[1]))
    #find samples with same labels and different labels from nearest prototype nprot
    idx_same, idx_diff=np.where(i_dist_same==nprot)[0], np.where(i_dist_diff==nprot)[0]
    get_grad_same=sklvq.distances.SquaredEuclidean.gradient(sklvq.distances.SquaredEuclidean(), 
                                                           data=data_unlearn.iloc[idx_same], model=glvq2, i_prototype=nprot)
    get_grad_diff=sklvq.distances.SquaredEuclidean.gradient(sklvq.distances.SquaredEuclidean(), 
                                                           data=data_unlearn.iloc[idx_diff], model=glvq2, i_prototype=nprot)
    if len(idx_same)>0:
        if len(idx_same)==1:
        #unlearn_grad_same=get_grad.iloc[ind_same].copy()*data_unlearn.iloc[ind_same]
            unlearn_grad_same=get_grad_same*data_unlearn.iloc[idx_same]
            prot_init_corr=np.reshape(data_unlearn.iloc[idx_same], (1,glvq2.prototypes_.shape[1]))
        else:
            #dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot-1]
            unlearn_grad_same=get_grad_same.mean()*data_unlearn.iloc[idx_same].mean()
            prot_init_corr=np.reshape(data_unlearn.iloc[idx_same].mean(), (1,glvq.prototypes_.shape[1]))
    else:
        unlearn_grad_same=np.zeros(np.shape(prot))
    if len(idx_diff)>=1:
        if len(idx_diff)==1:
            unlearn_grad_diff=get_grad_diff*data_unlearn.iloc[idx_diff]
            prot_init_corr=np.reshape(data_unlearn.iloc[idx_diff], (1,model.prototypes_.shape[1]))
        else:
            unlearn_grad_diff=get_grad_diff.mean()*data_unlearn.iloc[idx_diff].mean()
            prot_init_corr=np.reshape(data_unlearn.iloc[idx_diff].mean(), (1,model.prototypes_.shape[1]))
    else:
        unlearn_grad_diff=np.zeros(np.shape(prot))
    unlearn_grad_diff=np.reshape(unlearn_grad_diff, (1,model.prototypes_.shape[1]))
    unlearn_grad_same=np.reshape(unlearn_grad_same, (1,model.prototypes_.shape[1]))
    updated_prots[nprot,:]=updated_prots[nprot,:]+(unlearn_grad_diff-unlearn_grad_same)*step_size

    
updated_prots


array([[ 0.83412654,  0.61885251,  0.85853045,  0.80492566,  0.49978456,
         0.86544582,  0.90632635,  0.99921421,  0.53197351, -0.00408364,
         0.6729556 ,  0.01680739,  0.67541665,  0.61288198, -0.08035148,
         0.4357498 ,  0.34395362,  0.60759699,  0.05622605,  0.14494943,
         0.89749128,  0.64174949,  0.91408406,  0.83747475,  0.59304593,
         0.909243  ,  0.96383663,  1.08084683,  0.67158121,  0.66241929],
       [ 0.91652956,  0.51228006,  0.91774465,  0.88743588,  0.45877197,
         0.6942594 ,  0.82488101,  0.94005693,  0.36855421, -0.04227734,
         0.75183931,  0.00810667,  0.71731785,  0.67551184, -0.08126933,
         0.35432604,  0.32679013,  0.5216551 ,  0.03463349,  0.06047538,
         0.96181972,  0.56931399,  0.957778  ,  0.91341686,  0.47264875,
         0.68878198,  0.78632477,  0.93363128,  0.49818327,  0.33332916],
       [-0.80305662, -0.50648848, -0.80210326, -0.76656761, -0.35596072,
        -0.53429342, -0.64475344, -0.76607053, -0

In [28]:
print(np.mean(np.abs(updated_prots-glvq2_partial1.prototypes_)))
print(root_mean_squared_error(updated_prots, glvq2_partial1.prototypes_))

0.07606804825282593
0.08813563758592198


In [34]:
%autoreload 2
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq

updated_prots3=unlearn_sample_effect_glvq(glvq2_copy, zXtrain.iloc[unlearn_indices], Ytrain[unlearn_indices])
print(np.mean(np.abs(updated_prots3-glvq2_partial1.prototypes_)))
print(root_mean_squared_error(updated_prots3, glvq2_partial1.prototypes_))

0.07606804825282593
0.08813563758592198


In [35]:
updated_prots0=unlearn_sample_effect_glvq(glvq_copy, zXtrain.iloc[unlearn_indices], Ytrain[unlearn_indices])
print(np.mean(np.abs(updated_prots0-glvq_partial1.prototypes_)))
print(root_mean_squared_error(updated_prots0, glvq_partial1.prototypes_))

0.062163542577256765
0.06671836889215102


In [37]:
glvq2_copy.prototypes_-glvq2.prototypes_

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [38]:
glvq2_copy.set_prototypes(updated_prots3)
glvq2_copy.prototypes_-glvq2.prototypes_

array([[-1.23057899e-01,  5.66902680e-02, -1.18636543e-01,
        -1.08913487e-01,  9.76545984e-03,  6.08340848e-02,
        -3.92540991e-02, -6.74734619e-02,  3.53273497e-02,
         4.55045235e-02, -3.70138845e-02,  1.55350818e-02,
        -1.86865707e-02, -5.19125578e-02,  1.47688190e-02,
         6.51414425e-02,  1.38598810e-02,  8.23625121e-02,
         3.16801006e-03,  8.54543076e-02, -1.26753340e-01,
        -5.01368659e-04, -1.26738420e-01, -1.12341293e-01,
         1.93919944e-02,  8.03538220e-02,  2.14517999e-02,
        -1.55580220e-02,  2.31552222e-02,  2.47248668e-01],
       [-7.34756964e-02, -2.46910671e-02, -8.88004211e-02,
        -6.69974946e-02, -1.98315996e-02, -7.98677772e-02,
        -1.18287998e-01, -1.29333791e-01, -8.52490829e-02,
         3.33305904e-02,  2.14857903e-02,  1.98782924e-03,
         1.24305008e-02, -1.28366963e-02,  1.36794788e-04,
         3.70486788e-03, -3.11203694e-03,  2.09772441e-03,
         2.79982453e-02, -4.98078101e-03, -9.53066745e-

In [43]:
nc=1
computed_dist2[xind_same==nc,0]

array([  7.82739674,   7.76303076,  11.60721146,  11.73831507,
        34.26171676,  18.34092298,  68.87643467,  25.21516451,
        23.77769774,  21.69573735,  17.11055724,  25.37569319,
        11.51790095,   8.39323377,  16.5794547 ,  16.42511683,
         9.38446947,  75.92694454,  40.4525906 ,   5.586925  ,
        15.69559452,  10.19613375, 189.82963667,  43.22317018,
        14.86784919,   6.30744172,  59.38882098,  11.30935241,
        34.78766467,   8.82687311,  17.82701574,  17.20572733,
        11.67041961,  18.87691459,  22.35932903,  10.45855795,
        27.15048794,   6.99043143,  19.12354763,   6.51567984,
        50.30436456,  26.70296941,   8.40649754,   9.2724548 ,
        20.95190051,  18.10383067,  28.349912  ,   9.17342035,
        14.26855171,  16.03028546,  13.94477513,   5.16768641,
        17.36594851,   8.87080472,  39.84334072, 351.14780169,
        18.36227518,  10.34818684,  34.47465588,  35.31809235,
        19.19756266,  23.55817081,  41.16229766,  36.83

In [33]:


step_size=model.get_params()['solver_params']['step_size'][0]
for nprot in nprots:
    get_grad=sklvq.distances.SquaredEuclidean.gradient(sklvq.distances.SquaredEuclidean(), 
                                                       data=zXtrain.iloc[unlearn_indices], model=glvq2, i_prototype=nprot)
    if xind_same==nprot:
        dist_same=computed_dist[]
    #if nprot==0:
    #    dist_same, dist_diff=computed_dist[xind_same==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot+1]
    #else:
    #    dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot-1]
    prot=np.reshape(model.prototypes_[nprot], (1,model.prototypes_.shape[1]))
    ind_same,ind_diff=np.where(dist_same<dist_diff)[0], np.where(dist_same>dist_diff)[0]
    if nprot>=0:
        if len(ind_same)>=1:            
            if len(ind_same)==1:
                unlearn_grad_same=get_grad.iloc[ind_same].copy()*data_unlearn.iloc[ind_same]
                prot_init_corr=np.reshape(data_unlearn.iloc[ind_same], (1,model.prototypes_.shape[1]))
            else:
                unlearn_grad_same=get_grad.iloc[ind_same].mean()* data_unlearn.iloc[ind_same].mean()
                prot_init_corr=np.reshape(data_unlearn.iloc[ind_same].mean(), (1,model.prototypes_.shape[1]))
        else:
            unlearn_grad_same=np.zeros(np.shape(prot))
        #updated_prots[nprot,:]-=unlearn_grad_same*step_size
        if len(ind_diff)>=1:
            if len(ind_diff)==1:
                unlearn_grad_diff=get_grad.iloc[ind_diff].copy()*data_unlearn.iloc[ind_diff]
                prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff], (1,model.prototypes_.shape[1]))
            else:
                unlearn_grad_diff=get_grad.iloc[ind_diff].mean()*data_unlearn.iloc[ind_diff].mean()
                prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff].mean(), (1,model.prototypes_.shape[1]))
           # prot-=prot_init_corr
        else:
            unlearn_grad_diff=np.zeros(np.shape(prot))
        unlearn_grad_diff=np.reshape(unlearn_grad_diff, (1,model.prototypes_.shape[1]))
        unlearn_grad_same=np.reshape(unlearn_grad_same, (1,model.prototypes_.shape[1]))
        updated_prots[nprot,:]=updated_prots[nprot,:]+(unlearn_grad_diff-unlearn_grad_same)*step_size

(4, 30)

array([2, 0, 3, 2, 3, 2, 2, 3, 1, 2, 1, 2, 0, 0, 2, 2, 3, 3, 3, 3, 3, 2,
       1, 3, 2, 2, 3, 2, 2, 0, 1, 3, 3, 0, 2, 2, 3, 3, 2, 2, 2, 3, 1, 0,
       2, 2, 0, 2, 0, 2, 2, 2, 2, 3, 2, 2, 2, 3, 1, 3, 0, 2, 1, 2, 1, 2,
       2, 3, 3, 2, 1, 3, 2, 0, 2, 3, 1, 1, 3, 0, 2, 3, 3, 3, 2, 1, 2, 3,
       3, 3, 2, 1, 2, 2, 3, 1, 0, 0, 3, 3, 1, 2, 0, 0, 2, 3, 3, 2, 3, 3,
       0, 3, 2, 3, 2, 0, 2, 0, 2, 3, 1, 0, 2, 2, 0, 3, 0, 0, 2, 0, 1, 2,
       0, 0, 0, 1, 3, 2, 2, 1, 3, 0, 0, 0, 2, 2, 3, 1, 3, 2, 2, 2, 3, 0,
       0, 1, 2, 3, 2, 3, 3, 2, 0, 2, 3, 0, 0, 2, 0, 2, 1, 3, 2, 3, 2, 3,
       2, 0, 3, 2, 0, 3, 2, 3, 3, 2, 3, 1, 2, 3, 3, 2, 1, 1, 0, 3, 0, 2,
       2, 1, 1, 1, 2, 3, 3, 3, 3, 3, 0, 2, 2, 2, 1, 3, 2, 1, 3, 3, 1, 3,
       1, 3, 2, 3, 0, 0, 3, 1, 2, 2, 3, 0, 2, 2, 2, 3, 1, 3, 2, 1, 1, 0,
       2, 2, 3, 0, 3, 1, 0, 3, 3, 2, 0, 3, 0, 1, 1, 2, 1, 1, 2, 2, 1, 0,
       1, 0, 2, 1, 2, 2, 3, 3, 2, 2, 2, 1, 0, 0, 2, 3, 2, 2, 0, 2, 0, 2,
       2, 3, 2, 3, 0, 0, 1, 3, 2, 1, 2, 2, 1, 1, 0,

## Unlearn feature influences (relevance matrix)

In [9]:
#unlearn_ftr_ind=np.array(np.random.randint(0, len(features)- 1, size=4).tolist())
#relearn_ftr_ind=np.setdiff1d(list(range(0, len(features)- 1)), unlearn_ftr_ind)
#unlearn_features=trainset[features[unlearn_ftr_ind]].copy()
#relearn_features=trainset[features[relearn_ftr_ind]].copy()
#unlearn_features2=unlearn_features.copy()
#relearn_features2=relearn_features.copy()
#unlearn_features2.to_csv(datapath+'unlearn_features.csv', index=False)
#relearn_features2.to_csv(datapath+'relearn_features.csv', index=False)
#glvq_partial2=GLVQ(distance_type="squared-euclidean", activation_type="identity",solver_type="wgd",
#                solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},random_state=42)
#mu_new2, std_new2= relearn_features.mean(), relearn_features.std()
#zXretrain2, zXretest2=(relearn_features-mu_new2)/std_new2, (Xtest[features[relearn_ftr_ind]]-mu_new2)/std_new2
gmlvq=model = GMLVQ(
    distance_type="adaptive-squared-euclidean", activation_type="identity", relevance_n_components=1,
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    random_state=42)
gmlvq_copy=GMLVQ(
    distance_type="adaptive-squared-euclidean", activation_type="identity", relevance_n_components=1, 
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    random_state=42)

gmlvq_partial2=GMLVQ(
    distance_type="adaptive-squared-euclidean", activation_type="identity", relevance_n_components=1, 
    solver_type="wgd", solver_params={"max_runs": 5, "k": 3, "step_size": np.array([0.05])},
    random_state=42)

gmlvq.fit(zXtrain, Ytrain)
gmlvq_copy.fit(zXtrain, Ytrain)
est_train, est_test=gmlvq.predict(zXtrain), glvq.predict(zXtest)
est_train_copy, est_test_copy=glvq_copy.predict(zXtrain), glvq_copy.predict(zXtest)
gmlvq_partial2.fit(zXretrain, Ytrain[relearn_indices])
#Ytrain[unlearn_indices]

GMLVQ(activation_type='identity', random_state=42, relevance_n_components=1,
      solver_params={'k': 3, 'max_runs': 5, 'step_size': array([0.05])},
      solver_type='wgd')

In [ ]:
def unlearn_sample_rel_effect_glvq(model, unlearn_indices, labels_unlearn, data_unlearn):
    updated_prots, o_omega=model.prototypes_.copy(), model.omega_
    dist_func=sklvq.distances.AdaptiveSquaredEuclidean()
    step_size=model.get_params()['solver_params']['step_size'][0]
    model.prototypes_secure_=model.prototypes_.copy()
    
    computed_dist=dist_func(data_unlearn, model)
    # for prot update
    for nprot in model.prototypes_labels_:
        get_grad=sklvq.distances.AdaptiveSquaredEuclidean.gradient(sklvq.distances.AdaptiveSquaredEuclidean(), 
                                                           data=data_unlearn, model=model, i_prototype=nprot)
        if nprot==0:
            dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot+1]
        else:
            dist_same, dist_diff=computed_dist[labels_unlearn==nprot,nprot], computed_dist[labels_unlearn==nprot,nprot-1]
        prot=np.reshape(model.prototypes_[nprot], (1,model.prototypes_.shape[1]))
        ind_same,ind_diff=np.where(dist_same<dist_diff)[0], np.where(dist_same>dist_diff)[0]
        if nprot>=0:
            if len(ind_same)>=1:            
                if len(ind_same)==1:
                    unlearn_grad_same=get_grad_prot.iloc[ind_same].copy()*data_unlearn.iloc[ind_same]*o_omega
                    prot_init_corr=np.reshape(data_unlearn.iloc[ind_same], (1,model.prototypes_.shape[1]))
                else:
                    unlearn_grad_same=get_grad_prot.iloc[ind_same].mean()* data_unlearn.iloc[ind_same].mean()*o_omega
                    prot_init_corr=np.reshape(data_unlearn.iloc[ind_same].mean(), (1,model.prototypes_.shape[1]))
            else:
                unlearn_grad_same=np.zeros(np.shape(prot))
            if len(ind_diff)>=1:
                if len(ind_diff)==1:
                    unlearn_grad_diff=get_grad_prot.iloc[ind_diff].copy()*data_unlearn.iloc[ind_diff]*o_omega
                    prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff], (1,model.prototypes_.shape[1]))
                else:
                    unlearn_grad_diff=get_grad_prot.iloc[ind_diff].mean()*data_unlearn.iloc[ind_diff].mean()*o_omega
                    prot_init_corr=np.reshape(data_unlearn.iloc[ind_diff].mean(), (1,model.prototypes_.shape[1]))
            else:
                unlearn_grad_diff=np.zeros(np.shape(prot))
            unlearn_grad_diff=np.reshape(unlearn_grad_diff, (1,model.prototypes_.shape[1]))
            unlearn_grad_same=np.reshape(unlearn_grad_same, (1,model.prototypes_.shape[1]))
            updated_prots[nprot,:]=updated_prots[nprot,:]+(unlearn_grad_diff-unlearn_grad_same)*step_sizey
    model.prototypes_=updated_prots.copy()
    # for omega update
    get_grad_omega=sklvq.distances.AdaptiveSquaredEuclidean.gradient(sklvq.distances.AdaptiveSquaredEuclidean(), 
                                                           data=data_unlearn, model=model, i_prototype=nprot) 
    return updated_prots

In [19]:
get_grad=sklvq.distances.AdaptiveSquaredEuclidean.gradient(sklvq.distances.AdaptiveSquaredEuclidean(), 
                                                           data=zXretrain, model=gmlvq, i_prototype=0)
print(np.shape(get_grad[1,:]))

(60,)


In [23]:
gmlvq.feature_names_in_

array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error',
       'fractal dimension error', 'worst radius', 'worst texture',
       'worst perimeter', 'worst area', 'worst smoothness',
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype=object)

In [12]:
import sklvq
updated_prots2=np.empty((np.shape(glvq_copy.prototypes_)))
sedist=sklvq.distances.SquaredEuclidean()
og_computed_dist=sedist(zXtrain, glvq_copy)

dummy_prots=np.empty((np.shape(glvq_copy.prototypes_)))
step_size=glvq.get_params()['solver_params']['step_size'][0]
for nrun in range(0, glvq.get_params()['solver_params']['max_runs']):   
    for nprot in [0,1]:
        if nrun==0:
            init_prot=zXtrain.loc[Ytrain==nprot].mean().to_numpy()
            prot=init_prot.copy() # prot=glvq.prototypes_[nprot]
        xx, yy=np.diag(np.dot(zXtrain, zXtrain.T)), np.dot(prot, prot.T)
        xy=zXtrain @ prot[np.newaxis,:].T
        sqedist=np.sqrt(xx[:,np.newaxis]+yy-2*xy)
        prot=prot-step_size*np.mean(sqedist[Ytrain==nprot])
        ind=np.where(Ytrain!=nprot)[0]
        prot=prot+step_size*np.mean(sqedist[Ytrain!=nprot])
        dummy_prots[nprot,:]=prot
    

for nprot in [0,1]:
    prot=glvq.prototypes_[nprot]
    get_grad=sklvq.distances.SquaredEuclidean.gradient(sklvq.distances.SquaredEuclidean(), data=zXtrain, model=glvq, i_prototype=nprot)
    Ytrain_unlean=Ytrain[unlearn_indices]
    ind=np.where(Ytrain_unlean==nprot)[0]
    unlearn_grad=get_grad.iloc[np.array(unlearn_indices)[ind]]
    updated_prots2[nprot,:]=prot+glvq.get_params()['solver_params']['step_size'][0]*np.mean(unlearn_grad)
    ind=np.where(Ytrain_unlean!=nprot)[0]
    unlearn_grad=get_grad.iloc[np.array(unlearn_indices)[ind]]
    updated_prots2[nprot,:]=prot-glvq.get_params()['solver_params']['step_size'][0]*np.mean(unlearn_grad)

In [13]:
print(updated_prots2)
glvq_partial1.prototypes_#.shape#['step_size']

[[ 0.87293766  0.44396609  0.89096223  0.8337366   0.37680151  0.68068261
   0.84039614  0.96649393  0.36490269 -0.17605636  0.61668816 -0.10330969
   0.59572578  0.57273897 -0.19659172  0.25010395  0.2232273   0.41684012
  -0.08248124 -0.04738728  0.9400087   0.52415936  0.95242209  0.87091666
   0.46104714  0.6915692   0.82516844  0.98526883  0.49723381  0.28329427]
 [        nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan
          nan         nan         nan         nan         nan         nan]]


array([[ 0.97845225,  0.56054294,  0.99619323,  0.93896495,  0.48029859,
         0.78417955,  0.94397219,  1.07043527,  0.46891956, -0.07196016,
         0.72158176,  0.00586869,  0.70023569,  0.67793518, -0.09288575,
         0.35413013,  0.32807956,  0.52102936,  0.02142091,  0.05682168,
         1.04544963,  0.6404666 ,  1.05741664,  0.97607599,  0.56410861,
         0.79527343,  0.92901351,  1.08926761,  0.60118285,  0.38741938],
       [-0.6109798 , -0.3814035 , -0.62347019, -0.58974096, -0.30117315,
        -0.51139047, -0.61816894, -0.68464123, -0.304954  ,  0.01200618,
        -0.46775802, -0.04198911, -0.45578996, -0.4344092 ,  0.02412286,
        -0.25875642, -0.2453001 , -0.35947582, -0.03158236, -0.08368636,
        -0.65486396, -0.42782337, -0.66370938, -0.61381404, -0.35370454,
        -0.51370958, -0.60670987, -0.6929001 , -0.37528728, -0.27059955]])

In [25]:
print(zXtrain.shape)
print(np.shape(glvq.prototypes_))
print(np.shape(glvq.get_variables()))

(381, 30)
(2, 30)
(60,)


In [19]:
relevance=np.diag(gmlvq.omega_*gmlvq.omega_.T)/np.trace(gmlvq.omega_*gmlvq.omega_.T)
ftr_rel=pd.DataFrame(data=np.reshape(relevance, (-1,len(features))), 
                     columns=features)
#print(ftr_rel[ftr_rel.max()])
most_rel_ftr, least_rel_ftr=features[relevance==np.max(relevance)], features[relevance==np.min(relevance)]
print(most_rel_ftr, least_rel_ftr)

NameError: name 'gmlvq' is not defined

In [154]:
adapt_sqe_dist=sklvq.distances.SquaredEuclidean()
computed_dist=adapt_sqe_dist(zXtrain.iloc[unlearn_samples], gmlvq_copy)
labels_unlearn=Ytrain[unlearn_samples]
grads_same, grads_diff=[],[]
for lab in [0,1]:
    if lab==0:
        dist_same, dist_diff=computed_dist[labels_unlearn==lab,lab], computed_dist[labels_unlearn==lab,lab+1]
    else:
        dist_same, dist_diff=computed_dist[labels_unlearn==lab,lab], computed_dist[labels_unlearn==lab,lab-1]
    gradients=np.multiply(gmlvq_copy.lambda_, _gradient_same(dist_same, dist_diff))
    grads_same.append(gradients)
    gradients=np.dot(_gradient_diff(dist_same, dist_diff), gmlvq_copy.lambda_)# np.array([True]*np.sum(labels_unlearn==lab)))
    grads_diff.append(gradients)

ValueError: operands could not be broadcast together with shapes (30,30) (8,) 

In [ ]:
sklvq.discriminants.RelativeDistance.

In [128]:
from sklvq.discriminants import DiscriminantBaseClass
class InverseRelativeDistance(DiscriminantBaseClass):
    def __call__(self, dist_same: np.ndarray, dist_diff: np.ndarray) -> np.ndarray:
        # dist_same = distance to prototype with same label as X.
        # dist_diff = distance to prototype with different label as X.
        return (dist_same - dist_diff) / (dist_same + dist_diff)

    def gradient(
        self, dist_same: np.ndarray, dist_diff: np.ndarray, winner_same: bool) -> np.ndarray:
        # Winner_same is an boolean flag to indicate if the considered prototype has the same or
        # a different label compared to the considered X.
        if winner_same:
            return _gradient_same(dist_same, dist_diff)
        return _gradient_diff(dist_same, dist_diff)


# Gradient depends on if the label is the same or different
def _gradient_same(dist_same: np.ndarray, dist_diff: np.ndarray) -> np.ndarray:
    return -2*dist_diff / (dist_same + dist_diff) ** 2


# Gradient depends on if the label is the same or different
def _gradient_diff(dist_same: np.ndarray, dist_diff: np.ndarray) -> np.ndarray:
    return 2*dist_same / (dist_same + dist_diff) ** 2

In [149]:
omega=gmlvq_copy.omega_.copy()
lambda_new=gmlvq_copy.omega_.T*gmlvq_copy.omega_
omega=(grads_same[0]+grads_diff[0])
prot_neg_L=-0.05*(dist_diff[1]/((dist_diff[0]+dist_same[0])**2))*np.dot(
    lambda_new,(samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[0]).T)
prot_neg_K=0.05*(dist_same[0]/((dist_diff[0]+dist_same[0])**2))*np.dot(
    lambda_new,(samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[1]).T)
gmlvq_copy.prototypes_[0]+=(prot_neg_L+prot_neg_K)

omega+=(grads_same[1]+grads_diff[1])
prot_pos_L=-0.05*(dist_diff[0]/((dist_diff[1]+dist_same[1])**2))*np.dot(
    lambda_new,(samples_to_unlean.loc[labels_to_unlearn==1]-gmlvq_copy.prototypes_[1]).T)
prot_pos_K=0.05*(dist_same[1]/((dist_diff[1]+dist_same[1])**2))*np.dot(
    lambda_new,(samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[0]).T)
gmlvq_copy.prototypes_[1]+=(prot_pos_L+prot_pos_K)
gmlvq_copy.omega_=omega.copy()


ValueError: operands could not be broadcast together with shapes (30,30) (8,) (30,30) 

In [130]:
samples_to_unlean=zXtrain.iloc[unlearn_samples].copy()
labels_to_unlearn=Ytrain[unlearn_samples].copy()
lambda_new=gmlvq_copy.omega_*gmlvq_copy.omega_.T
(grads_diff[0]/(grads_diff[0]+grads_same[0])**2)*lambda_new*(
    samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[0]).T

ValueError: operands could not be broadcast together with shapes (8,) (30,30) 

In [117]:
grads_diff

[array([0.17584577, 0.16397241, 0.10406393, 6.0049628 , 0.13772069,
        0.18855362, 0.09750045, 0.12675661]),
 array([0.00506333, 0.00440107])]

In [145]:
print(np.shape(np.dot(lambda_new,(samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[0]).T)))
print(np.shape((grads_diff[0]/(grads_diff[0]+grads_same[0])**2)*np.dot(
    lambda_new,(samples_to_unlean.loc[labels_to_unlearn==0]-gmlvq_copy.prototypes_[0]).T)))

(30, 8)
(30, 8)
